In [1]:
import urllib.request
from pathlib import Path
import torch

import torch.nn as nn

import tokenizers
import logging


In [2]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s"
)

def enable_logging():
    logging.disable(logging.NOTSET)

def disable_logging():
    logging.disable(logging.CRITICAL)


# enable_logging()
disable_logging()

In [3]:
def download_shakespeare_text():
    path = Path("../../data/processed/shakespeare/shakespeare.txt")
    if not path.is_file():
        path.parent.mkdir(parents=True, exist_ok=True)
        url = "https://homl.info/shakespeare"
        with urllib.request.urlopen(url) as response:
            text_content = response.read().decode("utf-8")
        path.write_text(text_content, 'utf-8')
        return text_content
    else:
        return path.read_text()

shakespeare_text = download_shakespeare_text()

In [5]:
tokenzier_path = Path("tokenziers") / "NLP_shakespeare.json"
tokenizer = tokenizers.Tokenizer(tokenizers.models.BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.ByteLevel()
tokenizer.decoder = tokenizers.decoders.ByteLevel()
tokenizer_trainer = tokenizers.trainers.BpeTrainer(
    special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"], 
    vocab_size=1000
)
if tokenzier_path.exists() and tokenzier_path.is_file():
    logging.info(f"Tokenizer found at {tokenzier_path}. Loading...")
    tokenizer = tokenizers.Tokenizer.from_file(str(tokenzier_path))
else:
    logging.info(f"No tokenzier found at {tokenzier_path}. Training one and saving.")
    tokenizer.train_from_iterator([shakespeare_text], tokenizer_trainer)
    tokenizer.enable_padding(pad_id=tokenizer.token_to_id("[PAD]"), pad_token="[PAD]")
    tokenzier_path.parent.mkdir(parents=True, exist_ok=True)
    tokenizer.save(str(tokenzier_path))

In [35]:
print("Vocab size:", tokenizer.get_vocab_size())
print("PAD ID:", tokenizer.token_to_id("[PAD]"))
print("UNK ID:", tokenizer.token_to_id("[UNK]"))

encoded = tokenizer.encode("All:\nWe know't, we know't.")

print("Tokens:", encoded.tokens)
print("IDs:", encoded.ids)
print("Decoded:", tokenizer.decode(encoded.ids))

Vocab size: 1000
PAD ID: 3
UNK ID: 0
Tokens: ['ĠA', 'll', ':', 'Ċ', 'We', 'Ġknow', "'t", ',', 'Ġwe', 'Ġknow', "'t", '.']
IDs: [446, 87, 13, 68, 481, 318, 480, 9, 145, 318, 480, 11]
Decoded:  All:
We know't, we know't.


In [36]:
class TokenRNN(nn.Module):
    def __init__(self, vocab_size:int,hidden_size:int = 64, gru_num_layers:int = 1, embedding_dim:int=16, padding_idx:int = 0):
        super().__init__()
        
        self.embedding_layer = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim,padding_idx=padding_idx)

        self.encoder = nn.GRU(
            input_size=embedding_dim, 
            hidden_size=hidden_size, 
            num_layers = gru_num_layers, 
            batch_first=True
        )

        self.classifier = nn.Linear(in_features=hidden_size, out_features=vocab_size)


    def forward(self, X):
        logging.info(f"X: {X.shape}")
        embeddings = self.embedding_layer(X)
        logging.info(f"embeddings: {embeddings.shape}")
        encoder_output, _ = self.encoder(embeddings)
        logging.info(f"encoder_output: {encoder_output.shape}")
        output = self.classifier(encoder_output)
        logging.info(f"output: {output.shape}")
        output = output.permute(0, 2,1)
        logging.info(f"permutted output: {output.shape}")
        return output
        


class TokenShakespeareDataset(torch.utils.data.Dataset):
    def __init__(self, text:str, tokenzier:tokenizers.Tokenizer, window_length:int = 8, shift_window:int = 1):
        super().__init__()
        self.window_length = window_length
        self.shift_window = shift_window
        self.tokenzier = tokenzier

        self.encoded_text = torch.Tensor(tokenizer.encode(text).ids)

        self.encoded_sets = self.encoded_text.unfold(dimension=0, size=self.window_length, step=self.shift_window)

    def __len__(self):
        return len(self.encoded_sets) - 1

    def __getitem__(self, index):
        return self.encoded_sets[index], self.encoded_sets[index+1]

In [37]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TokenRNN(
    vocab_size=tokenizer.get_vocab_size(),
    hidden_size = 64, 
    gru_num_layers = 1, 
    embedding_dim=16, 
    padding_idx = tokenizer.token_to_id("[PAD]")
).to(device=device)

In [82]:
generator = torch.Generator().manual_seed(42)
dataset = TokenShakespeareDataset(shakespeare_text, tokenizer, window_length = 32, shift_window=1)
train_ds, test_ds = torch.utils.data.random_split(dataset, (0.8, 0.2), generator=generator)

LOG_ITEM_AT = 1000
BATCH_SIZE = 128
EPOCHS = 100
NUM_PROCESSES = 0
train_loader, test_loader = torch.utils.data.DataLoader(
    train_ds, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    pin_memory=True, 
    num_workers=NUM_PROCESSES), torch.utils.data.DataLoader(
    test_ds, 
    batch_size=BATCH_SIZE, 
    pin_memory=True, 
    num_workers=NUM_PROCESSES)
LR = 3e-5
loss_fn = torch.nn.CrossEntropyLoss()
optim = torch.optim.AdamW(params=model.parameters(), lr=LR)

In [83]:
len(train_loader)

346

In [84]:

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0
    test_loss = 0
    for idx, (x, y) in enumerate(train_loader, start=1):
        x = x.to(device=device, dtype=torch.long)
        y = y.to(device=device, dtype=torch.long)
        output = model(x)
        loss = loss_fn(output, y)
        train_loss += loss.item()
        optim.zero_grad()
        loss.backward()
        optim.step()
        if idx == 1 or idx == len(train_loader) or idx % LOG_ITEM_AT == 0:
            logging.info(f"[TRAIN] | [EPOCH: {epoch:04d}] | [STEP: {idx:05d} / {len(train_loader):05d}] | [Train: {train_loss/idx:5f}] | [Test: {test_loss/len(test_loader):.5f}]")
            print(f"[TRAIN] | [EPOCH: {epoch:04d}] | [STEP: {idx:05d} / {len(train_loader):05d}] | [Train: {train_loss/idx:5f}] | [Test: {test_loss/len(test_loader):.5f}]")
            
    model.eval()
    with torch.inference_mode():
        for idx, (x, y) in enumerate(test_loader, start=1):
            x = x.to(device=device, dtype=torch.long)
            y = y.to(device=device, dtype=torch.long)
            output = model(x)
            loss = loss_fn(output, y)
            test_loss += loss.item()
            if idx == 1 or idx == len(train_loader) or idx % LOG_ITEM_AT == 0:
                logging.info(f"[TEST] | [EPOCH: {epoch:04d}] | [STEP: {idx:05d} / {len(test_loader):05d}] | [Train: {train_loss/len(train_loader):5f}] | [Test: {test_loss/idx:.5f}]")
                print(f"[TEST] | [EPOCH: {epoch:04d}] | [STEP: {idx:05d} / {len(test_loader):05d}] | [Train: {train_loss/len(train_loader):5f}] | [Test: {test_loss/idx:.5f}]")


    logging.info(f"[EPOCH: {epoch:04d}] | [Train: {train_loss/len(train_loader):5f}] | [Test: {test_loss/len(test_loader):.5f}]")
    print(f"[EPOCH: {epoch:04d}] | [Train: {train_loss/len(train_loader):5f}] | [Test: {test_loss/len(test_loader):.5f}]")
        

[TRAIN] | [EPOCH: 0001] | [STEP: 00001 / 00346] | [Train: 5.579950] | [Test: 0.00000]
[TRAIN] | [EPOCH: 0001] | [STEP: 00346 / 00346] | [Train: 5.518369] | [Test: 0.00000]
[TEST] | [EPOCH: 0001] | [STEP: 00001 / 00087] | [Train: 5.518369] | [Test: 5.64040]
[EPOCH: 0001] | [Train: 5.518369] | [Test: 5.63840]
[TRAIN] | [EPOCH: 0002] | [STEP: 00001 / 00346] | [Train: 5.508949] | [Test: 0.00000]
[TRAIN] | [EPOCH: 0002] | [STEP: 00346 / 00346] | [Train: 5.510213] | [Test: 0.00000]
[TEST] | [EPOCH: 0002] | [STEP: 00001 / 00087] | [Train: 5.510213] | [Test: 5.63647]
[EPOCH: 0002] | [Train: 5.510213] | [Test: 5.63516]
[TRAIN] | [EPOCH: 0003] | [STEP: 00001 / 00346] | [Train: 5.493834] | [Test: 0.00000]
[TRAIN] | [EPOCH: 0003] | [STEP: 00346 / 00346] | [Train: 5.507135] | [Test: 0.00000]
[TEST] | [EPOCH: 0003] | [STEP: 00001 / 00087] | [Train: 5.507135] | [Test: 5.63494]
[EPOCH: 0003] | [Train: 5.507135] | [Test: 5.63360]
[TRAIN] | [EPOCH: 0004] | [STEP: 00001 / 00346] | [Train: 5.486358] | [Te

In [90]:
shakespeare_model_path = Path("models") / "shakespeare_tokenizer_limit_1.pt"

shakespeare_model_path.parent.mkdir(parents=True, exist_ok=True)

In [91]:
# torch.save(model.state_dict(), str(shakespeare_model_path))
model.load_state_dict(torch.load(str(shakespeare_model_path)))

<All keys matched successfully>

In [92]:
input_text = "All:\nWe know't, we know't."

output_text = []
encoded_input = torch.Tensor(tokenizer.encode(input_text).ids)
encoded_input = encoded_input.unsqueeze(0).to(device=device, dtype=torch.long) # added the batch dimension
for idx in range(50):
    output_token = model(encoded_input)[:,:, -1] # complete batch dimension fetched, then comes the sequence and then lastly the logits
    token_id = torch.argmax(output_token, dim=1)
    token = tokenizer.decode([token_id.item()])[0]
    token_id = token_id.to(device=device)
    encoded_input = torch.cat(
            [encoded_input, token_id.unsqueeze(1)],
            dim=1
        )
    output_text.append(token)


print(f"Input: {input_text}")
print(f"Output: {input_text + ''.join(output_text)}")
    

Input: All:
We know't, we know't.
Output: All:
We know't, we know't.

AGL:
I   ,
A,  ,  '  gay.

K  :
I'    oy,
A,  , 


In [93]:
input_text = "All:\nWe know't, we know't."

temperature = 0.7
top_k = 10
max_new_tokens = 500
output_text = []

encoded_input = torch.tensor(
    tokenizer.encode(input_text).ids,
    dtype=torch.long,
    device=device
).unsqueeze(0)

model.eval()

with torch.inference_mode():
    for idx in range(max_new_tokens):
        # Keep the context length consistent with training.
        model_input = encoded_input[:, -8:]
        # Model output: [B, vocab_size, sequence_length]
        raw = model(model_input)
        # Get logits for the last time step.
        logits = raw[:, :, -1]
        # Apply temperature.
        logits = logits / temperature
        # Select top-k logits.
        top_logits, top_indices = torch.topk(
            logits,
            k=top_k,
            dim=-1
        )
        # Convert top-k logits into probabilities.
        top_probs = torch.softmax(top_logits, dim=-1)
        # Sample an index within the top-k candidates.
        sampled_idx = torch.multinomial(
            top_probs,
            num_samples=1
        )
        # Map sampled index to the actual vocabulary token ID.
        token_id = top_indices.gather(
            dim=-1,
            index=sampled_idx
        )
        # Append generated token to input.
        encoded_input = torch.cat(
            [encoded_input, token_id],
            dim=1
        )
        # Decode the generated token.
        token = tokenizer.decode([token_id.item()])
        output_text.append(token)
print(f"Input: {input_text}")
print(f"Output: {input_text + ''.join(output_text)}")

Input: All:
We know't, we know't.
Output: All:
We know't, we know't.

ANTIGONUS:
May from my coister;
And I am too more than a master,
Are you like it beating,
That thou dost thou wish;
The mispronges to-day,
And to the king and the fight
Of a came of his mortal crown.

First Senator:
I pray thee, I am a boy,
The senate, and the body,
And in this sorrow, I would have not, I'll make me leave my father
Against the singing, to make a while,
And all the field of this duty.

PROSPERO:
The gown of his silver'st,
But, if he is, and intain.

KING EDWARD IV:
I'll tell you hence, and he's dead.

BENVOLIO:
Why, I will not,
That, I will not be not a body
To the mortal merry.

BUCKINGHAM:
O, I have no more than you.

GLOUCESTER:
More than the king's sithed
They have turn to the corrined.

BENVOLIO:
O, I have done, and not to be.

PETRUCHIO:
I do not bend it, for I know not.

QUEEN:
I would have a touch'd in my father's death,
And that I have to the city
And all my father, and sold a man,
And, my go

In [32]:
raw.shape

torch.Size([1, 134, 32])

In [33]:
raw[:,:,-1].shape

torch.Size([1, 134])

In [37]:
raw[:, -1,:].shape

torch.Size([1, 32])